# Lab 12 — Modelo de detecção de fraude

## Objetivo

Este laboratório encerra o pipeline com um modelo preditivo simples de regressão logística.

O modelo utilizará valor da transação, risk score, credit score e segmento para estimar a probabilidade de uma transação ser fraudulenta.

Como apenas 1,83% das transações são fraudes, a avaliação não será baseada somente em acurácia. Também serão analisados:

- AUC-ROC;
- AUC-PR;
- matriz de confusão;
- precisão;
- recall;
- especificidade;
- F1-score;
- coeficientes da regressão.

O modelo tem finalidade didática e não deve ser tratado como solução pronta para produção.

In [38]:
from pathlib import Path
import math
import pandas as pd
import pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "silver").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "silver").exists():
            pasta_projeto = pasta_pai
            break

arquivo_silver = (
    pasta_projeto
    / "dados"
    / "silver"
    / "transactions_enriched.parquet"
)

assert arquivo_silver.exists(), "Arquivo Silver não encontrado."

caminho_silver_spark = arquivo_silver.as_posix()

print("PySpark:", pyspark.__version__)
print("Silver:", caminho_silver_spark)

PySpark: 4.2.0
Silver: C:/BigData/bigdata-curso-gabriel/dados/silver/transactions_enriched.parquet


In [39]:
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab12_modelo_fraude")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Modo:", spark.sparkContext.master)

Spark: 4.2.0
Modo: local[2]


In [40]:
df = spark.read.parquet(caminho_silver_spark)

resumo_base = (
    df
    .agg(
        F.count("*").alias("transacoes"),
        F.sum(
            F.col("is_fraud").cast("integer")
        ).alias("fraudes")
    )
    .collect()[0]
)

taxa_fraude = (
    resumo_base["fraudes"]
    / resumo_base["transacoes"]
)

print("Transações:", resumo_base["transacoes"])
print("Fraudes:", resumo_base["fraudes"])
print(f"Taxa de fraude: {100 * taxa_fraude:.2f}%")

Transações: 100000
Fraudes: 1833
Taxa de fraude: 1.83%


In [41]:
df_ml = (
    df
    .select(
        "transaction_id",
        "amount",
        "risk_score",
        "credit_score",
        "segment",
        "is_fraud"
    )
    .withColumn(
        "label",
        F.col("is_fraud").cast("double")
    )
)

df_ml.groupBy("label").count().orderBy("label").show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0|98167|
|  1.0| 1833|
+-----+-----+



In [42]:
indexador_segmento = StringIndexer(
    inputCol="segment",
    outputCol="segment_idx",
    handleInvalid="keep"
)

montador_features = VectorAssembler(
    inputCols=[
        "amount",
        "risk_score",
        "credit_score",
        "segment_idx"
    ],
    outputCol="features",
    handleInvalid="skip"
)

regressao_logistica = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

pipeline = Pipeline(
    stages=[
        indexador_segmento,
        montador_features,
        regressao_logistica
    ]
)

print("Pipeline definido.")

Pipeline definido.


In [43]:
treino, teste = df_ml.randomSplit(
    [0.8, 0.2],
    seed=42
)

treino = treino.cache()
teste = teste.cache()

total_treino = treino.count()
total_teste = teste.count()

print("Treino:", total_treino)
print("Teste:", total_teste)
print("Total:", total_treino + total_teste)

Treino: 79901
Teste: 20099
Total: 100000


In [44]:
distribuicao_split = (
    treino
    .groupBy("label")
    .count()
    .withColumn("amostra", F.lit("Treino"))
    .unionByName(
        teste
        .groupBy("label")
        .count()
        .withColumn("amostra", F.lit("Teste"))
    )
    .withColumn(
        "percentual",
        F.round(
            100 * F.col("count")
            / F.sum("count").over(
                __import__(
                    "pyspark"
                ).sql.Window.partitionBy("amostra")
            ),
            2
        )
    )
    .orderBy("amostra", "label")
)

distribuicao_split.show()

+-----+-----+-------+----------+
|label|count|amostra|percentual|
+-----+-----+-------+----------+
|  0.0|19729|  Teste|     98.16|
|  1.0|  370|  Teste|      1.84|
|  0.0|78438| Treino|     98.17|
|  1.0| 1463| Treino|      1.83|
+-----+-----+-------+----------+



In [45]:
modelo = pipeline.fit(treino)

print("Modelo treinado com sucesso.")

Modelo treinado com sucesso.


In [46]:
predicoes = modelo.transform(teste).cache()

print("Predições geradas:", predicoes.count())

Predições geradas: 20099


In [47]:
avaliador_roc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

avaliador_pr = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

auc_roc = avaliador_roc.evaluate(predicoes)
auc_pr = avaliador_pr.evaluate(predicoes)

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AUC-PR:  {auc_pr:.4f}")

AUC-ROC: 0.7675
AUC-PR:  0.0756


In [48]:
contagens = {
    (
        int(linha["label"]),
        int(linha["prediction"])
    ): linha["count"]

    for linha in (
        predicoes
        .groupBy("label", "prediction")
        .count()
        .collect()
    )
}

tn = contagens.get((0, 0), 0)
fp = contagens.get((0, 1), 0)
fn = contagens.get((1, 0), 0)
tp = contagens.get((1, 1), 0)

matriz_confusao = pd.DataFrame(
    [
        [tn, fp],
        [fn, tp]
    ],
    index=[
        "Real: não fraude",
        "Real: fraude"
    ],
    columns=[
        "Previsto: não fraude",
        "Previsto: fraude"
    ]
)

matriz_confusao

,Previsto: não fraude,Previsto: fraude
Real: não fraude,19729,0
Real: fraude,370,0


In [49]:
def divisao_segura(numerador, denominador):
    return (
        numerador / denominador
        if denominador != 0
        else 0.0
    )


acuracia = divisao_segura(
    tp + tn,
    tp + tn + fp + fn
)

precisao = divisao_segura(
    tp,
    tp + fp
)

recall = divisao_segura(
    tp,
    tp + fn
)

especificidade = divisao_segura(
    tn,
    tn + fp
)

f1_score = divisao_segura(
    2 * precisao * recall,
    precisao + recall
)

acuracia_baseline = divisao_segura(
    tn + fp,
    tp + tn + fp + fn
)

metricas_modelo = pd.DataFrame({
    "Métrica": [
        "Acurácia do modelo",
        "Acurácia do baseline sem fraude",
        "Precisão",
        "Recall ou sensibilidade",
        "Especificidade",
        "F1-score",
        "AUC-ROC",
        "AUC-PR"
    ],
    "Resultado": [
        acuracia,
        acuracia_baseline,
        precisao,
        recall,
        especificidade,
        f1_score,
        auc_roc,
        auc_pr
    ]
})

metricas_modelo["Resultado"] = (
    metricas_modelo["Resultado"].round(4)
)

metricas_modelo

,Métrica,Resultado
0,Acurácia do modelo,0.9816
1,Acurácia do baseline sem fraude,0.9816
2,Precisão,0.0000
3,Recall ou sensibilidade,0.0000
4,Especificidade,1.0000
5,F1-score,0.0000
6,AUC-ROC,0.7675
7,AUC-PR,0.0756


In [50]:
from pyspark.ml.functions import vector_to_array

predicoes_exibicao = (
    predicoes
    .withColumn(
        "probability_array",
        vector_to_array("probability")
    )
    .withColumn(
        "probabilidade_nao_fraude",
        F.round(
            F.col("probability_array")[0],
            4
        )
    )
    .withColumn(
        "probabilidade_fraude",
        F.round(
            F.col("probability_array")[1],
            4
        )
    )
)

(
    predicoes_exibicao
    .select(
        "transaction_id",
        "amount",
        "risk_score",
        "credit_score",
        "segment",
        "label",
        "prediction",
        "probabilidade_nao_fraude",
        "probabilidade_fraude"
    )
    .orderBy(
        F.col("probabilidade_fraude").desc()
    )
    .show(15, truncate=False)
)

+--------------+------------------+-----------------+------------+---------+-----+----------+------------------------+--------------------+
|transaction_id|amount            |risk_score       |credit_score|segment  |label|prediction|probabilidade_nao_fraude|probabilidade_fraude|
+--------------+------------------+-----------------+------------+---------+-----+----------+------------------------+--------------------+
|71011         |30.276138532885103|99.96640567199488|771         |High-Risk|0.0  |0.0       |0.8544                  |0.1456              |
|16164         |13.344220227937448|99.95101367456148|623         |High-Risk|0.0  |0.0       |0.8551                  |0.1449              |
|11912         |30.920752038407446|99.74531673673296|704         |High-Risk|0.0  |0.0       |0.8553                  |0.1447              |
|61045         |30.38677372885751 |99.34696794281088|821         |High-Risk|0.0  |0.0       |0.8553                  |0.1447              |
|19418         |102.

In [51]:
modelo_indexador = modelo.stages[0]

mapeamento_segmentos = pd.DataFrame({
    "segment": modelo_indexador.labels,
    "segment_idx": range(
        len(modelo_indexador.labels)
    )
})

mapeamento_segmentos

,segment,segment_idx
0,Premium,0
1,Standard,1
2,High-Risk,2


In [52]:
modelo_lr = modelo.stages[-1]

nomes_features = [
    "amount",
    "risk_score",
    "credit_score",
    "segment_idx"
]

coeficientes = list(modelo_lr.coefficients)

tabela_coeficientes = pd.DataFrame({
    "Variável": nomes_features,
    "Coeficiente": coeficientes,
    "Odds ratio por unidade": [
        math.exp(coeficiente)
        for coeficiente in coeficientes
    ]
})

tabela_coeficientes["Coeficiente"] = (
    tabela_coeficientes["Coeficiente"].round(6)
)

tabela_coeficientes["Odds ratio por unidade"] = (
    tabela_coeficientes[
        "Odds ratio por unidade"
    ].round(6)
)

tabela_coeficientes

,Variável,Coeficiente,Odds ratio por unidade
0,amount,-0.000198,0.999802
1,risk_score,0.016723,1.016864
2,credit_score,0.000058,1.000058
3,segment_idx,1.184904,3.270373


In [53]:
incrementos = {
    "amount": 100,
    "risk_score": 10,
    "credit_score": 50,
    "segment_idx": 1
}

efeitos = []

for nome, coeficiente in zip(
    nomes_features,
    coeficientes
):
    incremento = incrementos[nome]

    efeitos.append({
        "Variável": nome,
        "Incremento analisado": incremento,
        "Multiplicador das chances": round(
            math.exp(coeficiente * incremento),
            4
        )
    })

efeitos_coeficientes = pd.DataFrame(efeitos)
efeitos_coeficientes

,Variável,Incremento analisado,Multiplicador das chances
0,amount,100,0.9804
1,risk_score,10,1.1820
2,credit_score,50,1.0029
3,segment_idx,1,3.2704


In [54]:
from pyspark.ml.functions import vector_to_array

predicoes_scores = (
    predicoes
    .withColumn(
        "probabilidade_fraude",
        vector_to_array("probability")[1]
    )
    .select(
        "transaction_id",
        "label",
        "probabilidade_fraude"
    )
    .cache()
)

print(
    "Transações avaliadas:",
    predicoes_scores.count()
)

Transações avaliadas: 20099


In [55]:
limiares = [
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.075,
    0.10,
    0.125
]

resultados_limiares = []

for limiar in limiares:
    resumo = (
        predicoes_scores
        .agg(
            F.sum(
                F.when(
                    (F.col("label") == 1)
                    & (
                        F.col("probabilidade_fraude")
                        >= limiar
                    ),
                    1
                ).otherwise(0)
            ).alias("tp"),

            F.sum(
                F.when(
                    (F.col("label") == 0)
                    & (
                        F.col("probabilidade_fraude")
                        >= limiar
                    ),
                    1
                ).otherwise(0)
            ).alias("fp"),

            F.sum(
                F.when(
                    (F.col("label") == 1)
                    & (
                        F.col("probabilidade_fraude")
                        < limiar
                    ),
                    1
                ).otherwise(0)
            ).alias("fn"),

            F.sum(
                F.when(
                    (F.col("label") == 0)
                    & (
                        F.col("probabilidade_fraude")
                        < limiar
                    ),
                    1
                ).otherwise(0)
            ).alias("tn")
        )
        .collect()[0]
    )

    tp_l = int(resumo["tp"])
    fp_l = int(resumo["fp"])
    fn_l = int(resumo["fn"])
    tn_l = int(resumo["tn"])

    precisao_l = divisao_segura(
        tp_l,
        tp_l + fp_l
    )

    recall_l = divisao_segura(
        tp_l,
        tp_l + fn_l
    )

    especificidade_l = divisao_segura(
        tn_l,
        tn_l + fp_l
    )

    f1_l = divisao_segura(
        2 * precisao_l * recall_l,
        precisao_l + recall_l
    )

    alertas = tp_l + fp_l

    resultados_limiares.append({
        "Limiar": limiar,
        "Alertas": alertas,
        "% da base em alerta": (
            100 * alertas / total_teste
        ),
        "Verdadeiros positivos": tp_l,
        "Falsos positivos": fp_l,
        "Falsos negativos": fn_l,
        "Precisão": precisao_l,
        "Recall": recall_l,
        "Especificidade": especificidade_l,
        "F1-score": f1_l
    })

tabela_limiares = pd.DataFrame(
    resultados_limiares
)

tabela_limiares_exibicao = tabela_limiares.copy()

colunas_percentuais = [
    "% da base em alerta",
    "Precisão",
    "Recall",
    "Especificidade",
    "F1-score"
]

tabela_limiares_exibicao[
    colunas_percentuais
] = tabela_limiares_exibicao[
    colunas_percentuais
].round(4)

tabela_limiares_exibicao

,Limiar,Alertas,% da base em alerta,Verdadeiros positivos,Falsos positivos,Falsos negativos,Precisão,Recall,Especificidade,F1-score
0,0.010,10611,52.7937,317,10294,53,0.0299,0.8568,0.4782,0.0577
1,0.020,5015,24.9515,245,4770,125,0.0489,0.6622,0.7582,0.0910
2,0.030,3548,17.6526,218,3330,152,0.0614,0.5892,0.8312,0.1113
3,0.040,2166,10.7767,165,2001,205,0.0762,0.4459,0.8986,0.1301
4,0.050,1254,6.2391,112,1142,258,0.0893,0.3027,0.9421,0.1379
5,0.075,763,3.7962,82,681,288,0.1075,0.2216,0.9655,0.1447
6,0.100,429,2.1344,62,367,308,0.1445,0.1676,0.9814,0.1552
7,0.125,155,0.7712,21,134,349,0.1355,0.0568,0.9932,0.0800


In [56]:
melhor_f1 = tabela_limiares.loc[
    tabela_limiares["F1-score"].idxmax()
]

pd.DataFrame([melhor_f1]).round(4)

,Limiar,Alertas,% da base em alerta,Verdadeiros positivos,Falsos positivos,Falsos negativos,Precisão,Recall,Especificidade,F1-score
6,0.1,429.0,2.1344,62.0,367.0,308.0,0.1445,0.1676,0.9814,0.1552


In [57]:
predicoes.unpersist(blocking=True)
treino.unpersist(blocking=True)
teste.unpersist(blocking=True)

spark.stop()

print("Caches liberados.")
print("Sessão Spark encerrada.")
print("Lab 12 executado com sucesso.")

Caches liberados.
Sessão Spark encerrada.
Lab 12 executado com sucesso.


## Limitações

O modelo desenvolvido possui caráter didático e apresenta limitações que impedem sua utilização direta em produção:

1. **Base sintética:** os dados foram gerados artificialmente e não representam toda a diversidade, a evolução e a complexidade das fraudes reais.

2. **Evento desbalanceado:** somente 1,83% das transações são fraudulentas. Esse desbalanceamento fez com que o limiar padrão de 50% classificasse todas as operações como não fraude.

3. **Quantidade limitada de variáveis:** o modelo utilizou apenas valor da transação, risk score, credit score e segmento. Informações como dispositivo, localização, frequência recente, comportamento histórico e relacionamento entre contas poderiam melhorar a capacidade preditiva.

4. **Codificação ordinal do segmento:** a transformação de `segment` em `segment_idx` estabeleceu uma ordem numérica entre Premium, Standard e High-Risk. Essa abordagem pressupõe distâncias semelhantes entre as categorias, o que constitui uma simplificação metodológica.

5. **Escalas diferentes:** as variáveis não foram padronizadas. Por isso, a magnitude dos coeficientes não pode ser utilizada diretamente como medida comparável de importância.

6. **Separação aleatória:** treino e teste foram definidos por divisão aleatória. Em uma aplicação real, seria preferível uma validação temporal, treinando o modelo com períodos anteriores e avaliando-o em dados posteriores.

7. **Escolha do limiar:** o limiar de 10% foi escolhido a partir do desempenho no próprio conjunto de teste e entre um número limitado de alternativas. Em uma implementação adequada, o limiar deveria ser definido em uma amostra de validação independente.

8. **Ausência de custos de negócio:** não foram atribuídos custos específicos aos falsos positivos e falsos negativos. A escolha operacional do limiar depende do custo de revisar uma transação legítima e do prejuízo de deixar uma fraude passar.

9. **Ausência de validações adicionais:** não foram realizados validação cruzada, ajuste de hiperparâmetros, análise de estabilidade, avaliação de calibração das probabilidades ou monitoramento de mudanças no padrão dos dados.

Dessa forma, o modelo deve ser interpretado como uma demonstração do fluxo de machine learning, e não como um mecanismo definitivo para bloqueio automático de transações.

## Conclusão

Foi desenvolvido um modelo de regressão logística para estimar a probabilidade de fraude nas transações da TechPay. A base foi dividida em 79.901 registros para treinamento e 20.099 para teste, preservando no teste uma taxa de fraude de 1,84%.

O modelo apresentou AUC-ROC de 0,7675, indicando capacidade moderada de ordenar as transações segundo o risco. A AUC-PR foi de 0,0756, aproximadamente quatro vezes superior à prevalência de fraude, embora ainda reduzida em termos absolutos. Os coeficientes indicaram maior associação do risco com o segmento e o risk score, enquanto credit score e valor da transação apresentaram efeitos menores.

Com o limiar padrão de 50%, nenhuma das 370 fraudes do teste foi identificada. A acurácia de 98,16% apenas reproduziu o resultado de classificar todas as operações como não fraudulentas, produzindo recall e F1-score iguais a zero. Esse resultado demonstrou que a acurácia isolada é inadequada para avaliar eventos raros.

Entre os limiares testados, 10% apresentou o melhor F1-score. Nesse ponto, o modelo gerou 429 alertas, correspondentes a 2,13% do conjunto de teste, e identificou 62 fraudes. A precisão alcançou 14,45%, o recall foi de 16,76% e o F1-score chegou a 15,52%. Assim, o modelo conseguiu concentrar risco em uma parcela pequena das transações, mas ainda deixou de identificar a maioria das fraudes.

Conclui-se que a regressão logística possui utilidade como instrumento de priorização para uma fila de revisão, mas não apresenta desempenho suficiente para decisões automáticas. Uma aplicação real exigiria novas variáveis, tratamento mais adequado das categorias, definição do limiar com base nos custos do negócio, validação temporal e monitoramento contínuo.